In [0]:
%pip install databricks-langchain
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Common Ingestion Framework - Configuration
ingestion_config = [
    {
        "source_name": "sales_db",
        "source_type": "jdbc",        
        "status": "active"
    },
    {
        "source_name": "product_files",
        "source_type": "file",       
        "status": "active"
    },
    {
        "source_name": "customer_api",
        "source_type": "api",         
        "status": "active"
    }
]

print("Configuration loaded successfully!")
for config in ingestion_config:
    print(f"  Source: {config['source_name']} | Type: {config['source_type']}")

Configuration loaded successfully!
  Source: sales_db | Type: jdbc
  Source: product_files | Type: file
  Source: customer_api | Type: api


In [0]:
# Audit Log Setup
from datetime import datetime

audit_logs = []  # log storage

def write_audit_log(source_name, status, rows_count=0, error_message=None):
    log = {
        "source_name": source_name,
        "status": status,                          # success / failed
        "rows_ingested": rows_count,
        "start_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "error_message": error_message             # None if success
    }
    audit_logs.append(log)
    print(f" Audit Log → {source_name} | {status} | Rows: {rows_count}")

print(" Audit Log function ready!")

 Audit Log function ready!


In [0]:
# Failure Handling + Restart Logic

# failed_sources = remembers which sources failed
# So if we restart, we skip successful ones and retry only failed ones

failed_sources = []
successful_sources = []

def ingest_source(config):
    source_name = config["source_name"]
    source_type = config["source_type"]
    
    # Skip if already successfully ingested (restart logic)
    if source_name in successful_sources:
        print(f"⏭Skipping {source_name} — already ingested!")
        return

    try:
        print(f"Starting ingestion: {source_name} ({source_type})")
        
        # Simulate ingestion based on source type
        if source_type == "jdbc":
            rows = 100  # Simulating 100 rows from database
        elif source_type == "file":
            rows = 250  # Simulating 250 rows from file
        elif source_type == "api":
            rows = 75   # Simulating 75 rows from API
        
        # Mark as successful
        successful_sources.append(source_name)
        write_audit_log(source_name, "success", rows)

    except Exception as e:
        # If anything fails, record it
        failed_sources.append(source_name)
        write_audit_log(source_name, "failed", error_message=str(e))
        print(f"Failed: {source_name} → {str(e)}")

print("Failure handling + Restart logic ready!")

Failure handling + Restart logic ready!


In [0]:
# Main Framework Runner
# ingests them one by one

def run_ingestion_framework():
    print("Starting Common Ingestion Framework...")
    for config in ingestion_config:
        ingest_source(config)
    print("\n Ingestion Summary:")
    print(f" Successful: {len(successful_sources)} sources")
    print(f" Failed: {len(failed_sources)} sources")
    
    print("\n Full Audit Log:")
    for log in audit_logs:
        print(f"  → {log['source_name']} | {log['status']} | "
              f"Rows: {log['rows_ingested']} | Time: {log['start_time']}")

run_ingestion_framework()

Starting Common Ingestion Framework...
Starting ingestion: sales_db (jdbc)
 Audit Log → sales_db | success | Rows: 100
Starting ingestion: product_files (file)
 Audit Log → product_files | success | Rows: 250
Starting ingestion: customer_api (api)
 Audit Log → customer_api | success | Rows: 75

 Ingestion Summary:
 Successful: 3 sources
 Failed: 0 sources

 Full Audit Log:
  → sales_db | success | Rows: 100 | Time: 2026-04-20 18:54:58
  → product_files | success | Rows: 250 | Time: 2026-04-20 18:54:58
  → customer_api | success | Rows: 75 | Time: 2026-04-20 18:54:58


In [0]:
# Testing Restart Logic
# Simulate: what if API source failed last time?
# Restart should SKIP successful ones, RETRY only failed

print("Simulating a restart scenario...")
print("Pretending customer_api failed last run...\n")

# Remove api from successful, add to failed
successful_sources.remove("customer_api")
failed_sources.append("customer_api")

print(f"Successful sources: {successful_sources}")
print(f" Failed sources: {failed_sources}")
print("\nNow restarting framework — watch it skip successful ones!\n")

# Reset audit for clean view
audit_logs.clear()
run_ingestion_framework()

Simulating a restart scenario...
Pretending customer_api failed last run...

Successful sources: ['sales_db', 'product_files']
 Failed sources: ['customer_api']

Now restarting framework — watch it skip successful ones!

Starting Common Ingestion Framework...
⏭Skipping sales_db — already ingested!
⏭Skipping product_files — already ingested!
Starting ingestion: customer_api (api)
 Audit Log → customer_api | success | Rows: 75

 Ingestion Summary:
 Successful: 3 sources
 Failed: 1 sources

 Full Audit Log:
  → customer_api | success | Rows: 75 | Time: 2026-04-20 18:54:58


In [0]:
# LLMOps - Track AI Usage in Framework
import time

llm_ops_log = []

def track_llm_call(task_name, model="databricks-claude", tokens_used=0):
    log = {
        "task": task_name,
        "model": model,
        "tokens_used": tokens_used,
        "estimated_cost_usd": round(tokens_used * 0.000002, 4),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    llm_ops_log.append(log)
    print(f" LLMOps → Task: {task_name} | Tokens: {tokens_used} | Cost: ${log['estimated_cost_usd']}")

# Simulate LLM being used during ingestion
print(" Simulating LLM usage during ingestion...\n")
track_llm_call("generate_jdbc_query", tokens_used=150)
track_llm_call("parse_file_schema", tokens_used=200)
track_llm_call("map_api_response", tokens_used=175)

print("\n LLMOps Summary:")
total_tokens = sum(l["tokens_used"] for l in llm_ops_log)
total_cost = sum(l["estimated_cost_usd"] for l in llm_ops_log)
print(f"  Total Tokens Used: {total_tokens}")
print(f"  Total Estimated Cost: ${round(total_cost, 4)}")

 Simulating LLM usage during ingestion...

 LLMOps → Task: generate_jdbc_query | Tokens: 150 | Cost: $0.0003
 LLMOps → Task: parse_file_schema | Tokens: 200 | Cost: $0.0004
 LLMOps → Task: map_api_response | Tokens: 175 | Cost: $0.0003

 LLMOps Summary:
  Total Tokens Used: 525
  Total Estimated Cost: $0.001


In [0]:
# Create sample data directly with Spark
from pyspark.sql import Row
from datetime import datetime

# Create data directly in Spark
sales_data = [
    Row(order_id=1, customer_name="John", product="Laptop", amount=1200, order_date="2026-04-01"),
    Row(order_id=2, customer_name="Sara", product="Phone",  amount=800,  order_date="2026-04-02"),
    Row(order_id=3, customer_name="Mike", product="Tablet", amount=450,  order_date="2026-04-03"),
    Row(order_id=4, customer_name="Lisa", product="Watch",  amount=300,  order_date="2026-04-04"),
    Row(order_id=5, customer_name="Tom",  product="TV",     amount=1500, order_date="2026-04-05"),
]

df_sales = spark.createDataFrame(sales_data)
print("Sales data created!")
print(f"Total rows: {df_sales.count()}")
df_sales.show()

Sales data created!
Total rows: 5
+--------+-------------+-------+------+----------+
|order_id|customer_name|product|amount|order_date|
+--------+-------------+-------+------+----------+
|       1|         John| Laptop|  1200|2026-04-01|
|       2|         Sara|  Phone|   800|2026-04-02|
|       3|         Mike| Tablet|   450|2026-04-03|
|       4|         Lisa|  Watch|   300|2026-04-04|
|       5|          Tom|     TV|  1500|2026-04-05|
+--------+-------------+-------+------+----------+



In [0]:
# Save directly to Bronze Delta table
df_sales.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_sales_data")

print(" Data saved to Bronze Delta table!")
write_audit_log("product_files", "success", df_sales.count())

 Data saved to Bronze Delta table!
 Audit Log → product_files | success | Rows: 5


In [0]:
# API Ingestion → Bronze Delta Table
import json
api_response = [
    {"user_id": 1, "name": "John Doe",    "email": "john@example.com",  "city": "New York",  "company": "ABC Corp"},
    {"user_id": 2, "name": "Sara Smith",  "email": "sara@example.com",  "city": "London",    "company": "XYZ Ltd"},
    {"user_id": 3, "name": "Mike Jones",  "email": "mike@example.com",  "city": "Toronto",   "company": "DEF Inc"},
    {"user_id": 4, "name": "Lisa Brown",  "email": "lisa@example.com",  "city": "Sydney",    "company": "GHI Co"},
    {"user_id": 5, "name": "Tom Wilson",  "email": "tom@example.com",   "city": "Singapore", "company": "JKL LLC"},
]

# Convert to Spark dataframe 
df_api = spark.createDataFrame(api_response)
print(" API data loaded!")
print(f" Total rows: {df_api.count()}")
df_api.show()

# Save to Bronze Delta table
df_api.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_api_users")

print("API data saved to Bronze Delta table!")
write_audit_log("customer_api", "success", df_api.count())

 API data loaded!
 Total rows: 5
+---------+--------+----------------+----------+-------+
|     city| company|           email|      name|user_id|
+---------+--------+----------------+----------+-------+
| New York|ABC Corp|john@example.com|  John Doe|      1|
|   London| XYZ Ltd|sara@example.com|Sara Smith|      2|
|  Toronto| DEF Inc|mike@example.com|Mike Jones|      3|
|   Sydney|  GHI Co|lisa@example.com|Lisa Brown|      4|
|Singapore| JKL LLC| tom@example.com|Tom Wilson|      5|
+---------+--------+----------------+----------+-------+

API data saved to Bronze Delta table!
 Audit Log → customer_api | success | Rows: 5


In [0]:
# Save Audit Log to Bronze Delta Table

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define schema explicitly
schema = StructType([
    StructField("source_name",   StringType(), True),
    StructField("status",        StringType(), True),
    StructField("rows_ingested", IntegerType(), True),
    StructField("start_time",    StringType(), True),
    StructField("error_message", StringType(), True),
])

# Clean audit logs (replace None with "N/A")
clean_logs = []
for log in audit_logs:
    clean_logs.append((
        log["source_name"],
        log["status"],
        log["rows_ingested"],
        log["start_time"],
        log["error_message"] if log["error_message"] else "N/A"
    ))

df_audit = spark.createDataFrame(clean_logs, schema)
print("Audit Log Table:")
df_audit.show(truncate=False)

# Save to Delta table
df_audit.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_audit_log")

print("Audit log saved to Bronze Delta table!")

Audit Log Table:
+-------------+-------+-------------+-------------------+-------------+
|source_name  |status |rows_ingested|start_time         |error_message|
+-------------+-------+-------------+-------------------+-------------+
|customer_api |success|75           |2026-04-20 18:54:58|N/A          |
|product_files|success|5            |2026-04-20 18:55:14|N/A          |
|customer_api |success|5            |2026-04-20 18:55:17|N/A          |
+-------------+-------+-------------+-------------------+-------------+

Audit log saved to Bronze Delta table!


In [0]:
# Verify all Bronze tables exist
print("1️ bronze_sales_data:")
spark.sql("SELECT * FROM bronze_sales_data").show()

print("2️ bronze_api_users:")
spark.sql("SELECT * FROM bronze_api_users").show()

print("3️ bronze_audit_log:")
spark.sql("SELECT * FROM bronze_audit_log").show()

1️ bronze_sales_data:
+--------+-------------+-------+------+----------+
|order_id|customer_name|product|amount|order_date|
+--------+-------------+-------+------+----------+
|       1|         John| Laptop|  1200|2026-04-01|
|       2|         Sara|  Phone|   800|2026-04-02|
|       3|         Mike| Tablet|   450|2026-04-03|
|       4|         Lisa|  Watch|   300|2026-04-04|
|       5|          Tom|     TV|  1500|2026-04-05|
+--------+-------------+-------+------+----------+

2️ bronze_api_users:
+---------+--------+----------------+----------+-------+
|     city| company|           email|      name|user_id|
+---------+--------+----------------+----------+-------+
| New York|ABC Corp|john@example.com|  John Doe|      1|
|   London| XYZ Ltd|sara@example.com|Sara Smith|      2|
|  Toronto| DEF Inc|mike@example.com|Mike Jones|      3|
|   Sydney|  GHI Co|lisa@example.com|Lisa Brown|      4|
|Singapore| JKL LLC| tom@example.com|Tom Wilson|      5|
+---------+--------+----------------+----

In [0]:
# Explore Databricks Sample Datasets

# Check what sample data is available
spark.sql("SHOW SCHEMAS IN samples").show()

+------------------+
|      databaseName|
+------------------+
|       accuweather|
|         bakehouse|
|      healthverity|
|information_schema|
|           nyctaxi|
|               sec|
|         tpcds_sf1|
|      tpcds_sf1000|
|              tpch|
|      wanderbricks|
+------------------+



In [0]:
# See all tables in each dataset
print("=== NYCTAXI ===")
spark.sql("SHOW TABLES IN samples.nyctaxi").show()

print("=== BAKEHOUSE ===")
spark.sql("SHOW TABLES IN samples.bakehouse").show()

print("=== TPCH ===")
spark.sql("SHOW TABLES IN samples.tpch").show()

=== NYCTAXI ===
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
| nyctaxi|    trips|      false|
+--------+---------+-----------+

=== BAKEHOUSE ===
+---------+--------------------+-----------+
| database|           tableName|isTemporary|
+---------+--------------------+-----------+
|bakehouse|media_customer_re...|      false|
|bakehouse|media_gold_review...|      false|
|bakehouse|     sales_customers|      false|
|bakehouse|    sales_franchises|      false|
|bakehouse|     sales_suppliers|      false|
|bakehouse|  sales_transactions|      false|
+---------+--------------------+-----------+

=== TPCH ===
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|    tpch| customer|      false|
|    tpch| lineitem|      false|
|    tpch|   nation|      false|
|    tpch|   orders|      false|
|    tpch|     part|      false|
|    tpch| partsupp|      false|
|    tpch|   region|      false|
|    t

In [0]:
# Preview bakehouse data
print("=== SALES TRANSACTIONS ===")
spark.sql("SELECT * FROM samples.bakehouse.sales_transactions LIMIT 5").show()

print("=== SALES CUSTOMERS ===")
spark.sql("SELECT * FROM samples.bakehouse.sales_customers LIMIT 5").show()

print("=== SALES FRANCHISES ===")
spark.sql("SELECT * FROM samples.bakehouse.sales_franchises LIMIT 5").show()

=== SALES TRANSACTIONS ===
+-------------+----------+-----------+--------------------+--------------------+--------+---------+----------+-------------+----------------+
|transactionID|customerID|franchiseID|            dateTime|             product|quantity|unitPrice|totalPrice|paymentMethod|      cardNumber|
+-------------+----------+-----------+--------------------+--------------------+--------+---------+----------+-------------+----------------+
|      1002961|   2000253|    3000047|2024-05-14 12:17:...|  Golden Gate Ginger|       8|        3|        24|         amex| 378154478982993|
|      1003007|   2000226|    3000047|2024-05-10 23:10:...|Austin Almond Bis...|      36|        3|       108|   mastercard|2244626981238094|
|      1003017|   2000108|    3000047|2024-05-16 16:34:...|Austin Almond Bis...|      40|        3|       120|   mastercard|2490570234487424|
|      1003068|   2000173|    3000047|2024-05-02 04:31:...|         Pearly Pies|      28|        3|        84|         am

In [0]:
# Real Data Ingestion - Bakehouse Dataset


from datetime import datetime

audit_logs = []

def write_audit_log(source_name, status, rows_count=0, error_message=None):
    log = {
        "source_name": source_name,
        "status": status,
        "rows_ingested": rows_count,
        "start_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "error_message": error_message
    }
    audit_logs.append(log)
    print(f" Audit Log → {source_name} | {status} | Rows: {rows_count}")

# Ingest 1 - Sales Transactions (JDBC type)
print(" Ingesting sales_transactions...")
df_transactions = spark.sql("SELECT * FROM samples.bakehouse.sales_transactions")
df_transactions.write.format("delta").mode("overwrite").saveAsTable("bronze_transactions")
write_audit_log("sales_transactions", "success", df_transactions.count())

# Ingest 2 - Customers (API type)
print(" Ingesting sales_customers...")
df_customers = spark.sql("SELECT * FROM samples.bakehouse.sales_customers")
df_customers.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")
write_audit_log("sales_customers", "success", df_customers.count())

# Ingest 3 - Franchises (File type)
print(" Ingesting sales_franchises...")
df_franchises = spark.sql("SELECT * FROM samples.bakehouse.sales_franchises")
df_franchises.write.format("delta").mode("overwrite").saveAsTable("bronze_franchises")
write_audit_log("sales_franchises", "success", df_franchises.count())

print("\n All real data ingested to Bronze Delta tables!")

 Ingesting sales_transactions...
 Audit Log → sales_transactions | success | Rows: 3333
 Ingesting sales_customers...
 Audit Log → sales_customers | success | Rows: 300
 Ingesting sales_franchises...
 Audit Log → sales_franchises | success | Rows: 48

 All real data ingested to Bronze Delta tables!


In [0]:
# Update Agent Tools for Real Data


from langchain_core.tools import tool

@tool
def get_transactions(question: str) -> str:
    """Use this to get real sales transactions from bronze_transactions table"""
    df = spark.sql("""
        SELECT product, SUM(totalPrice) as revenue, 
               COUNT(*) as total_orders
        FROM bronze_transactions 
        GROUP BY product 
        ORDER BY revenue DESC 
        LIMIT 10
    """)
    return f"Top 10 Products by Revenue:\n{df.toPandas().to_string()}"

@tool
def get_customers(question: str) -> str:
    """Use this to get customer data from bronze_customers table"""
    df = spark.sql("""
        SELECT continent, country, COUNT(*) as total_customers
        FROM bronze_customers
        GROUP BY continent, country
        ORDER BY total_customers DESC
        LIMIT 10
    """)
    return f"Customers by Country:\n{df.toPandas().to_string()}"

@tool
def get_franchises(question: str) -> str:
    """Use this to get franchise data from bronze_franchises table"""
    df = spark.sql("SELECT * FROM bronze_franchises LIMIT 10")
    return f"Franchises:\n{df.toPandas().to_string()}"

print(" Real data tools created!")
print("get_transactions (3,333 rows)")
print("get_customers (300 rows)")
print("get_franchises (48 rows)")

 Real data tools created!
get_transactions (3,333 rows)
get_customers (300 rows)
get_franchises (48 rows)


In [0]:

# Mosaic Agent on Real Bakehouse Data

from databricks_langchain import ChatDatabricks
from langgraph.prebuilt import create_react_agent
import mlflow

mlflow.set_experiment("/Users/theepankumar72@gmail.com/mosaic_agent_llmops")
mlflow.langchain.autolog()

# Create agent with real data tools
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1
)

tools = [get_transactions, get_customers, get_franchises]

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful bakehouse business analyst. Use tools to fetch real data from Bronze Delta tables before answering."
)

@mlflow.trace
def ask_real_agent(question):
    print(f"\nQuestion: {question}")

    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    answer = response["messages"][-1].content
    print(f"Answer: {answer}")
    return answer

ask_real_agent("What are the top 3 best selling products by revenue?")
ask_real_agent("Which continent has the most customers?")
ask_real_agent("How many franchises do we have?")

/home/spark-ea601d74-0f26-4159-ae73-98/.ipykernel/6797/command-7039977996360706-526403743:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



Question: What are the top 3 best selling products by revenue?
Answer: The top 3 best-selling products by revenue are:

1. Golden Gate Ginger with $11,595 in revenue
2. Outback Oatmeal with $11,199 in revenue
3. Austin Almond Biscotti with $11,148 in revenue

These products have the highest revenue among all products, with Golden Gate Ginger being the best-selling product.

Question: Which continent has the most customers?
Answer: Based on the output, Oceania has the most customers, with 108 customers in Australia.

Question: How many franchises do we have?
Answer: We have 10 franchises. They are located in various cities across the US, Australia, and Japan, and have different sizes and supplier IDs.


'We have 10 franchises. They are located in various cities across the US, Australia, and Japan, and have different sizes and supplier IDs.'

[Trace(trace_id=tr-010165ee176b53197ab1655cbde600ef), Trace(trace_id=tr-766eccfe3149e8b0acbb759fafd637bf), Trace(trace_id=tr-f097745f4be94c1e0b3f351704ebb582)]